# 自作LLM教材 — Google Colab ランチャー（marimo）

このノートブックは **Colab上でmarimoを起動するためのブートストラップ** です。
marimoのリアクティブUI（スライダー等）を保ったまま、ColabのGPUを使えます。

## 使い方
1. **ランタイム → ランタイプのタイプを変更 → T4 GPU** を選択（ch03の学習で必要）
2. 上から順にセルを実行
3. 最後のセルで表示されるURLを **新しいタブ** で開くと marimo が起動します


## ① marimo と不足ライブラリをインストール
Colabには torch / numpy / matplotlib / tqdm が最初から入っているので、足りないものだけ入れます。

In [ ]:
!pip install -q -U marimo regex tiktoken

import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('[注意] GPUが無効です。ランタイム→ランタイムのタイプを変更→T4 GPU を選んでください（ch01/ch02はCPUでもOK）')

## ② 教材ファイルを取得
GitHubに push 済みなら `REPO_URL` を設定して clone します。
未設定の場合は、左のファイルパネルから `.py` ファイルや `codebot/` `data/` を手動アップロードしてください。

In [ ]:
import os

REPO_URL = ""  # 例: "https://github.com/<your-name>/deeplearning-llm.git"

if REPO_URL:
    !rm -rf /content/llm && git clone $REPO_URL /content/llm
    os.chdir('/content/llm')
    print('cwd =', os.getcwd())
    !ls -1 *.py
else:
    os.makedirs('/content/llm', exist_ok=True)
    os.chdir('/content/llm')
    print('REPO_URL が未設定です。')
    print('左のファイルパネルに 01_tokenizer.py / 02_attention.py / codebot/ / data/ をアップロードしてください。')

## ③（任意）ch03 の学習データを取得
事前学習・SFT を本書のフルスケールで動かす場合のみ実行します。
（ch01/ch02 だけなら不要です）

In [ ]:
import os
os.makedirs('codebot', exist_ok=True)
BASE = 'https://raw.githubusercontent.com/oreilly-japan/deep-learning-from-scratch-6/refs/heads/main/codebot'

# 学習済みトークナイザ・トークン化済みデータ・SFTデータ（リポジトリ同梱・軽量）
for name in ['merge_rules.pkl', 'tiny_codes.bin', 'tiny_codes_sft.json', 'tiny_codes.txt']:
    !wget -nc -q "$BASE/$name" -O "codebot/$name"

# 学習済みモデル（事前学習をスキップして生成だけ試したい人向け / 大きいので任意）
# !wget -nc -q https://huggingface.co/datasets/koki0702/zero-llm-data/resolve/main/codebot/model_pretrain.pt -O codebot/model_pretrain.pt

!ls -lh codebot/

## ④ marimo を起動（プロキシ経由）
実行後に表示されるURLを **新しいタブ** で開いてください。
（Colab内のインラインフレームだとWebSocketが不安定になりがちなため）

In [ ]:
import secrets, subprocess, time
from google.colab import output

PORT = 8888
TOKEN = secrets.token_urlsafe(16)

# 既存のmarimoプロセスがあれば停止
subprocess.run('pkill -f marimo', shell=True)
time.sleep(1)

# headless でバックグラウンド起動（カレントフォルダのノートブック一覧が開きます）
subprocess.Popen(
    ['marimo', 'edit', '--headless', '--host', '127.0.0.1',
     '--port', str(PORT), '--token-password', TOKEN],
    stdout=open('/content/marimo.log', 'w'), stderr=subprocess.STDOUT,
)
time.sleep(5)

proxy = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
print('=' * 60)
print('↓ このURLを新しいタブで開いてください')
print(f'{proxy}?access_token={TOKEN}')
print('=' * 60)
print('\n開いた後、01_tokenizer.py / 02_attention.py を選択してください。')

### うまくいかないとき
- **真っ白 / 接続できない**: 上のセルをもう一度実行（`pkill` で再起動されます）
- **ログ確認**: `!tail -n 40 /content/marimo.log`
- **ポート競合**: `PORT = 8889` などに変えて再実行
- **GPUを使いたい**: ランタイムをGPUにしてから①から実行し直し